# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset and inspect the metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show a summary of record sets and their fields (by @id)
print("Available record sets (with @id):")
for rs in metadata.record_sets:
    print(f"  - {rs['@id']} : {rs['name']} ({rs['description'] if 'description' in rs else ''})")
    if 'fields' in rs and len(rs['fields']):
        for field in rs['fields']:
            print(f"      - field @id: {field['@id']} | name: {field.get('name', '')} | description: {field.get('description', '')}")
    else:
        print("      - [No field descriptions found]")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** In this section we extract all available record sets and print their columns for further exploration.

In [ ]:
# Identify record set @id values programmatically:
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set {record_set_id} with columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Error extracting records for {record_set_id}: {e}")

# For further analysis & EDA, pick the main record set id (typically the largest table of proteins/quant data) for demonstration:
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df_main = dataframes[main_record_set_id]
else:
    main_record_set_id = None
    df_main = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Let's choose numeric fields relevant to protein quantification for demonstration. _Update the field `@id` and column name as appropriate for your analysis based on the above overview._

In [ ]:
import numpy as np

if df_main is not None and len(df_main):
    print(f"Available columns for main record set {main_record_set_id}:")
    print(df_main.columns.tolist())

    # Example: Try to pick a common protein quantification or count column
    # If not present, use the first numeric column
    numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to coerce strings to numerics if possible (for demonstration)
        possible_numeric_cols = [c for c in df_main.columns if df_main[c].str.replace('.','',1).str.isnumeric().any()] if df_main.columns.size else []
        for col in possible_numeric_cols:
            df_main[col] = pd.to_numeric(df_main[col], errors='coerce')
        numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
    else:
        print("No numeric fields found.")
        numeric_field = None

    # Filtering for demonstration: rows where the numeric column > 10 (or another value)
    if numeric_field:
        threshold = 10
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalizing the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if present
        group_fields = [col for col in df_main.columns if df_main[col].dtype==object and col != numeric_field]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df)
        else:
            print("No categorical group field found for grouping.")
    else:
        print("No numeric fields to analyze.")    
else:
    print("No data available in the main DataFrame to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_main[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(9, 5))
        sns.boxplot(y=df_main[numeric_field], x=df_main[group_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we accessed the FAIR-compliant Croissant schema, record sets, and fields by their `@id`.
- We loaded dataframes for each record set, explored column and record summaries, and performed basic EDA.
- With pandas and seaborn/matplotlib, we visualized numeric field distributions and demonstrated basic grouping.
- This workflow can be extended for thorough statistical, machine learning, or proteomic analysis.